## Goal

Develop a chart statistics table that summarizes the long-term chart performance of individual songs.

### Tasks

- Load cleaned chart history
- Load song table
- Link `song_id` to chart history
- Explore and define chart metrics
- Validate aggregation results
- Export chart statistics

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# Load Datasets

project_path = Path.cwd().parent
interim_path = project_path / 'data' / 'interim'

chart_history = pd.read_csv(interim_path/'uk_chart_history_clean.csv')
songs = pd.read_csv(interim_path/'songs.csv')

### Validate Datasets

The cleaned chart history and song table are loaded and validated before the song identifiers are linked.

In [3]:
songs.info()

<class 'pandas.DataFrame'>
RangeIndex: 36655 entries, 0 to 36654
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   song_id  36655 non-null  int64
 1   Song     36655 non-null  str  
 2   Artist   36655 non-null  str  
dtypes: int64(1), str(2)
memory usage: 859.2 KB


In [4]:
chart_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 210882 entries, 0 to 210881
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Song            210882 non-null  str  
 1   Artist          210837 non-null  str  
 2   Position        210882 non-null  int64
 3   Last Week       210882 non-null  str  
 4   Peak            210882 non-null  int64
 5   Weeks on Chart  210882 non-null  int64
 6   Week            210882 non-null  str  
dtypes: int64(3), str(4)
memory usage: 11.3 MB


In [5]:
chart_history = chart_history.merge(
    songs,
    on=['Song', 'Artist'],
    how='left'
)
chart_history = chart_history[
    ['song_id',
     'Song',
     'Artist',
     'Position',
     'Last Week',
     'Peak',
     'Weeks on Chart',
     'Week']
]

chart_history.head()

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
0,1.0,SAVE YOUR LOVE,RENEE AND RENATO,1,LW:1,1,11,2 January 1983- 8 January 1983
1,2.0,YOU CAN'T HURRY LOVE,PHIL COLLINS,2,LW:6,2,6,2 January 1983- 8 January 1983
2,3.0,A WINTER'S TALE,DAVID ESSEX,3,LW:7,3,5,2 January 1983- 8 January 1983
3,4.0,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,LW:8,4,9,2 January 1983- 8 January 1983
4,5.0,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983


### Data quality correction

Duplicate chart entries occurring within the same chart week were identified
during the chart analysis. These entries create artificial overlapping chart
runs and are excluded from subsequent statistical calculations.

In [6]:
songs_to_delete = [{'song_id': 13496.0,
                    'Position': 57,
                    'Last Week': 'LW:New',
                    'Week': '3 September 1995- 9 September 1995'},
                    {'song_id': 13496.0,
                     'Position': 76,
                     'Last Week': 'LW:57',
                     'Week': '10 September 1995- 16 September 1995'},
                    {'song_id': 13965.0,
                     'Position': 91,
                     'Last Week': 'LW:New',
                     'Week': '24 December 1995- 30 December 1995'},
                    {'song_id': 13966.0,
                     'Position': 61,
                     'Last Week': 'LW:New',
                     'Week': '17 December 1995- 23 December 1995'},
                    {'song_id': 9155.0,
                     'Position': 19,
                     'Last Week': 'LW:New',
                     'Week': '3 January 1993- 9 January 1993'},
                    {'song_id': 8803.0,
                     'Position': 21,
                     'Last Week': 'LW:New',
                     'Week': '18 August 1991- 24 August 1991'},
                    {'song_id': 9061.0,
                     'Position': 22,
                     'Last Week': 'LW:New',
                     'Week': '24 November 1991- 30 November 1991'},
                    {'song_id': 10319.0,
                     'Position': 24,
                     'Last Week': 'LW:New',
                     'Week': '31 January 1993- 6 February 1993'},
                    {'song_id': 11079.0,
                     'Position': 36,
                     'Last Week': 'LW:New',
                     'Week': '17 October 1993- 23 October 1993'} ,
                    {'song_id': 11002.0,
                     'Position': 29,
                     'Last Week': 'LW:New',
                     'Week': '19 September 1993- 25 September 1993'}]

affected_song_ids = {item['song_id'] for item in songs_to_delete}

print(f'Number of affected songs: {len(affected_song_ids)}')

Number of affected songs: 9


In [7]:
print(f'Chart history rows before correction: {len(chart_history)}')

Chart history rows before correction: 210882


In [8]:
for item in songs_to_delete:
    mask = ((chart_history['song_id'] == item['song_id']) &
            (chart_history['Position'] == item['Position']) &
            (chart_history['Last Week'] == item['Last Week']) &
            (chart_history['Week'] == item['Week']))
    
    chart_history = chart_history.loc[~mask].copy()

In [9]:
print(f'Chart history rows after correction: {len(chart_history)}')

Chart history rows after correction: 210872


In [10]:
chart_history.info()

<class 'pandas.DataFrame'>
Index: 210872 entries, 0 to 210881
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   song_id         210827 non-null  float64
 1   Song            210872 non-null  str    
 2   Artist          210827 non-null  str    
 3   Position        210872 non-null  int64  
 4   Last Week       210872 non-null  str    
 5   Peak            210872 non-null  int64  
 6   Weeks on Chart  210872 non-null  int64  
 7   Week            210872 non-null  str    
dtypes: float64(1), int64(3), str(4)
memory usage: 14.5 MB


## Explore Individual Song Histories

Selected songs are examined to identify meaningful chart metrics and recurring chart patterns.

These observations guide the design of the chart statistics table.

In [11]:
song_history = chart_history[
    (chart_history['Song'] == 'OUR HOUSE') &
    (chart_history['Artist'] == 'MADNESS')
]

song_history

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
4,5.0,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983
112,5.0,OUR HOUSE,MADNESS,13,LW:5,5,8,9 January 1983- 15 January 1983
208,5.0,OUR HOUSE,MADNESS,10,LW:13,5,9,16 January 1983- 22 January 1983
319,5.0,OUR HOUSE,MADNESS,21,LW:10,5,10,23 January 1983- 29 January 1983
433,5.0,OUR HOUSE,MADNESS,35,LW:21,5,11,30 January 1983- 5 February 1983
551,5.0,OUR HOUSE,MADNESS,53,LW:35,5,12,6 February 1983- 12 February 1983
671,5.0,OUR HOUSE,MADNESS,73,LW:53,5,13,13 February 1983- 19 February 1983
149375,5.0,OUR HOUSE,MADNESS,92,LW:RE,5,14,10 June 2012- 16 June 2012
150382,5.0,OUR HOUSE,MADNESS,99,LW:RE,5,15,19 August 2012- 25 August 2012


In [12]:
song_history = chart_history[
    (chart_history['Song'] == 'RUNNING UP THAT HILL') &
    (chart_history['Artist'] == 'KATE BUSH')
]

song_history

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
13596,2448.0,RUNNING UP THAT HILL,KATE BUSH,9,LW:New,9,1,11 August 1985- 17 August 1985
13691,2448.0,RUNNING UP THAT HILL,KATE BUSH,4,LW:9,4,2,18 August 1985- 24 August 1985
13789,2448.0,RUNNING UP THAT HILL,KATE BUSH,3,LW:4,3,3,25 August 1985- 31 August 1985
13891,2448.0,RUNNING UP THAT HILL,KATE BUSH,5,LW:3,3,4,1 September 1985- 7 September 1985
13994,2448.0,RUNNING UP THAT HILL,KATE BUSH,8,LW:5,3,5,8 September 1985- 14 September 1985
14101,2448.0,RUNNING UP THAT HILL,KATE BUSH,15,LW:8,3,6,15 September 1985- 21 September 1985
14209,2448.0,RUNNING UP THAT HILL,KATE BUSH,23,LW:15,3,7,22 September 1985- 28 September 1985
14319,2448.0,RUNNING UP THAT HILL,KATE BUSH,33,LW:23,3,8,29 September 1985- 5 October 1985
14435,2448.0,RUNNING UP THAT HILL,KATE BUSH,49,LW:33,3,9,6 October 1985- 12 October 1985
14547,2448.0,RUNNING UP THAT HILL,KATE BUSH,61,LW:49,3,10,13 October 1985- 19 October 1985


In [13]:
song_history = chart_history[
    chart_history['Song'] == 'BARBIE GIRL'
]

song_history

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
73193,16442.0,BARBIE GIRL,AQUA,2,LW:New,2,1,19 October 1997- 25 October 1997
73292,16442.0,BARBIE GIRL,AQUA,1,LW:2,1,2,26 October 1997- 1 November 1997
73392,16442.0,BARBIE GIRL,AQUA,1,LW:1,1,3,2 November 1997- 8 November 1997
73492,16442.0,BARBIE GIRL,AQUA,1,LW:1,1,4,9 November 1997- 15 November 1997
73592,16442.0,BARBIE GIRL,AQUA,1,LW:1,1,5,16 November 1997- 22 November 1997
73693,16442.0,BARBIE GIRL,AQUA,2,LW:1,1,6,23 November 1997- 29 November 1997
73794,16442.0,BARBIE GIRL,AQUA,3,LW:2,1,7,30 November 1997- 6 December 1997
73894,16442.0,BARBIE GIRL,AQUA,3,LW:3,1,8,7 December 1997- 13 December 1997
73994,16442.0,BARBIE GIRL,AQUA,3,LW:3,1,9,14 December 1997- 20 December 1997
74097,16442.0,BARBIE GIRL,AQUA,6,LW:3,1,10,21 December 1997- 27 December 1997


## Design Chart Metrics

The chart history is explored to define meaningful statistics that describe the long-term chart performance of individual songs.

Each metric is developed and validated individually before creating the final chart statistics table.

### Peak Position

The peak position represents the highest chart position reached by each song.

It is calculated as the minimum chart position across the complete chart history of a song.

In [14]:
peak_position = (
    chart_history
    .groupby('song_id')['Position']
    .min()
    .rename('peak_position')
)

peak_position.head()

song_id
1.0    1
2.0    1
3.0    2
4.0    4
5.0    5
Name: peak_position, dtype: int64

In [15]:
# Metric Validation

validate_song = chart_history['song_id'] == 5
chart_history.loc[validate_song,:]

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
4,5.0,OUR HOUSE,MADNESS,5,LW:5,5,7,2 January 1983- 8 January 1983
112,5.0,OUR HOUSE,MADNESS,13,LW:5,5,8,9 January 1983- 15 January 1983
208,5.0,OUR HOUSE,MADNESS,10,LW:13,5,9,16 January 1983- 22 January 1983
319,5.0,OUR HOUSE,MADNESS,21,LW:10,5,10,23 January 1983- 29 January 1983
433,5.0,OUR HOUSE,MADNESS,35,LW:21,5,11,30 January 1983- 5 February 1983
551,5.0,OUR HOUSE,MADNESS,53,LW:35,5,12,6 February 1983- 12 February 1983
671,5.0,OUR HOUSE,MADNESS,73,LW:53,5,13,13 February 1983- 19 February 1983
149375,5.0,OUR HOUSE,MADNESS,92,LW:RE,5,14,10 June 2012- 16 June 2012
150382,5.0,OUR HOUSE,MADNESS,99,LW:RE,5,15,19 August 2012- 25 August 2012


### Total Chart Weeks

The official total chart duration is derived from the cumulative
`Weeks on Chart` value provided in the original chart dataset.

This metric reflects the complete chart history of a song, including chart
weeks before the observation period beginning in 1983.

An additional metric (`observed_chart_weeks`) is calculated separately to
describe only the chart appearances contained within the observation period.

In [16]:
total_chart_weeks = (chart_history
                    .groupby('song_id')['Weeks on Chart']
                    .max()
                    .rename('total_chart_weeks')
                    )

print(f'Number of total chart weeks: {total_chart_weeks.loc[5]}')

Number of total chart weeks: 15


### Observed Chart Weeks

The number of observed chart weeks is calculated by counting all chart
appearances within the available dataset.

Unlike the official chart duration, this metric only reflects the observation
period covered by this project.

In [17]:
observed_chart_weeks = (chart_history
                        .groupby('song_id')
                        .size()
                        .rename('observed_chart_weeks')
                        )

print(f'Number of observed chart weeks: {observed_chart_weeks.loc[5]}')

Number of observed chart weeks: 9


### Weeks at Number One

The number of observed weeks a song reached the number one position is
calculated by counting all chart entries with a chart position equal to one.

This metric only reflects chart appearances contained within the observation
period covered by this project.

In [18]:
weeks_top_1 = (chart_history[chart_history['Position'] == 1]
                .groupby('song_id')
                .size()
                .rename('weeks_top_1')
                )

chart_history[chart_history['Position'] == 1].head()

,song_id,Song,Artist,Position,Last Week,Peak,Weeks on Chart,Week
0,1.0,SAVE YOUR LOVE,RENEE AND RENATO,1,LW:1,1,11,2 January 1983- 8 January 1983
100,2.0,YOU CAN'T HURRY LOVE,PHIL COLLINS,1,LW:2,1,7,9 January 1983- 15 January 1983
199,2.0,YOU CAN'T HURRY LOVE,PHIL COLLINS,1,LW:1,1,8,16 January 1983- 22 January 1983
299,38.0,DOWN UNDER,MEN AT WORK,1,LW:2,1,4,23 January 1983- 29 January 1983
399,38.0,DOWN UNDER,MEN AT WORK,1,LW:1,1,5,30 January 1983- 5 February 1983


In [19]:
print(f'Weeks at Number 1: {weeks_top_1.loc[2]}')

Weeks at Number 1: 2


### Weeks in Top 10

The number of observed weeks spent within the Top 10 is calculated by counting
all chart entries with positions from 1 to 10.

This metric only reflects chart appearances contained within the observation
period covered by this project.

In [20]:
weeks_top_10 = (chart_history[chart_history['Position'] <= 10]
                .groupby('song_id')
                .size()
                .rename('weeks_top_10')
                )
print(f'Weeks in Top 10: {weeks_top_10.loc[5]}')

Weeks in Top 10: 2


### Weeks in Top 50

The number of observed weeks spent within the Top 50 is calculated by counting
all chart entries with positions from 1 to 50.

This metric only reflects chart appearances contained within the observation
period covered by this project.

In [21]:
weeks_top_50 = (chart_history[chart_history['Position'] <= 50]
                .groupby('song_id')
                .size()
                .rename('weeks_top_50')
                )
print(f'Weeks in Top 50: {weeks_top_50.loc[5]}')

Weeks in Top 50: 5


## Create Chart Statistics Table

The individual chart metrics are combined into a single table using the unique
song identifier.

This table summarizes the chart performance of every song and serves as the
foundation for subsequent analyses and Spotify enrichment.

In [22]:
chart_statistics = songs.copy()
chart_statistics.head()

,song_id,Song,Artist
0,1,SAVE YOUR LOVE,RENEE AND RENATO
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS
2,3,A WINTER'S TALE,DAVID ESSEX
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE
4,5,OUR HOUSE,MADNESS


In [23]:
# Add Peak Position

chart_statistics = (chart_statistics
                    .merge(peak_position,
                           on='song_id',
                           how='left')
                    )

In [24]:
# Add Total Chart Weeks

chart_statistics = (chart_statistics
                    .merge(total_chart_weeks,
                           on='song_id',
                           how='left')
                    )

In [25]:
# Add Observed Chart Weeks

chart_statistics = (chart_statistics
                    .merge(observed_chart_weeks,
                           on='song_id',
                           how='left')
                    )

In [26]:
# Add Weeks at Number One

chart_statistics = (chart_statistics
                    .merge(weeks_top_1,
                           on='song_id',
                           how='left')
                    )

In [27]:
# Songs without number-one weeks receive a value of 0

chart_statistics['weeks_top_1'] = (chart_statistics['weeks_top_1']
                                   .fillna(0)
                                   .astype(int)
                                   )

In [28]:
# Add Weeks in Top 10

chart_statistics = (chart_statistics
                    .merge(weeks_top_10,
                           on='song_id',
                           how='left')
                    )

In [29]:
# Replace missing values of count-based metrics with 0.

chart_statistics['weeks_top_10'] = (chart_statistics['weeks_top_10']
                                   .fillna(0)
                                   .astype(int)
                                   )

In [30]:
# Add Weeks in Top 50

chart_statistics = (chart_statistics
                    .merge(weeks_top_50,
                           on='song_id',
                           how='left')
                    )

In [31]:
chart_statistics['weeks_top_50'] = (chart_statistics['weeks_top_50']
                                   .fillna(0)
                                   .astype(int)
                                   )

### Validate Chart Statistics

The completed chart statistics table is reviewed to verify that the calculated
metrics have been merged correctly.

In [32]:
chart_statistics

,song_id,Song,Artist,peak_position,total_chart_weeks,observed_chart_weeks,weeks_top_1,weeks_top_10,weeks_top_50
0,1,SAVE YOUR LOVE,RENEE AND RENATO,1,16,6,1,2,5
1,2,YOU CAN'T HURRY LOVE,PHIL COLLINS,1,16,11,2,6,9
2,3,A WINTER'S TALE,DAVID ESSEX,2,10,6,0,3,6
3,4,BEST YEARS OF OUR LIVES,MODERN ROMANCE,4,13,5,0,2,5
4,5,OUR HOUSE,MADNESS,5,15,9,0,2,5
...,...,...,...,...,...,...,...,...,...
36650,36651,JUMP,TYLA/GUNNA/SKILLIBENG,80,1,1,0,0,0
36651,36652,ART,TYLA,85,1,1,0,0,0
36652,36653,WILDFLOWER AND BARLEY,HOZIER/ALLISON RUSSELL,86,1,1,0,0,0
36653,36654,EMPI NOW,HOZIER,92,1,1,0,0,0


In [33]:
chart_statistics.info()

<class 'pandas.DataFrame'>
RangeIndex: 36655 entries, 0 to 36654
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   song_id               36655 non-null  int64
 1   Song                  36655 non-null  str  
 2   Artist                36655 non-null  str  
 3   peak_position         36655 non-null  int64
 4   total_chart_weeks     36655 non-null  int64
 5   observed_chart_weeks  36655 non-null  int64
 6   weeks_top_1           36655 non-null  int64
 7   weeks_top_10          36655 non-null  int64
 8   weeks_top_50          36655 non-null  int64
dtypes: int64(7), str(2)
memory usage: 2.5 MB


In [34]:
print(chart_statistics.shape)

(36655, 9)


In [35]:
## Export Chart Statistics

interim_path = project_path / 'data' / 'interim'

chart_statistics.to_csv(interim_path/'chart_statistics.csv', index=False)
